In [ ]:
import os
import time
import math
import logging
import requests
import random
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
load_dotenv()

DATA_EX_KEY = os.getenv("DATA_EX_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")
DB_PORT = os.getenv("DB_PORT", "3306")

if not DATA_EX_KEY:
    raise RuntimeError("환경변수 DATA_EX_KEY가 설정되어 있지 않습니다.")
if not (DB_HOST and DB_USER and DB_PASSWORD and DB_NAME):
    raise RuntimeError("MySQL 접속을 위한 환경변수(DB_HOST, DB_USER, DB_PASSWORD, DB_NAME)가 모두 설정되어 있어야 합니다.")

In [ ]:
# 크롤링 설정
base_url = "https://data.ex.co.kr/openapi/business/curStateStation"
per_page = 100
params = {"key": DATA_EX_KEY, "type": "json", "numOfRows": per_page, "pageNo": 1}

In [ ]:
# 원하는 컬럼 목록, 프레임 리스트 초기화
want_cols = ["routeCode", "serviceAreaCode", "routeName", "lpgYn", "gasolinePrice", "diselPrice", "lpgPrice", "svarAddr"]
frames = []
print(want_cols)

In [ ]:
# requests 세션 사용
rq = requests.Session()
rq.headers.update({"User-Agent": "MyCrawler/1.0 (+https://example.com)"})
def fetch_json(page, retries=2):
    params_local = params.copy()
    params_local["pageNo"] = page
    backoff = 1
    for attempt in range(retries + 1):
        try:
            rp = rq.get(base_url, params=params_local, timeout=10)
            rp.raise_for_status()
            return rp.json()
        except requests.RequestException as e:
            logging.warning("Request error page %s attempt %s: %s", page, attempt+1, e)
        except ValueError as e:
            logging.warning("JSON decode error page %s attempt %s: %s", page, attempt+1, e)
        time.sleep(backoff + random.random() * 0.5)
        backoff *= 2
    return None

In [ ]:
# 총 개수 확인 (안전하게 처리)
js = fetch_json(1)
if not js:
    raise RuntimeError("첫 페이지 호출 실패로 진행 불가")
total_count = js.get("count") or js.get("totalCount") or 0
if not total_count:
    logging.info("count 정보가 없거나 0입니다. 첫 페이지 데이터만 처리합니다.")
    total_pages = 1
else:
    total_pages = math.ceil(int(total_count) / per_page)
logging.info("총 항목: %s, 페이지 수: %s, 페이지당: %s", total_count, total_pages, per_page)

In [ ]:
# 첫 페이지 처리
def process_page_json(js, page):
    page_list = js.get("list") or []
    if not page_list:
        return None
    df_page = pd.json_normalize(page_list)
    df_page.columns = df_page.columns.str.strip()
    if "pageNo" not in df_page.columns or df_page["pageNo"].isna().all():
        df_page = df_page.copy()
        df_page["_page"] = page
    selected = [col for col in want_cols if col in df_page.columns]
    if "_page" in df_page.columns and "_page" not in selected:
        selected.append("_page")
    if not selected:
        logging.info("원하는 컬럼이 페이지에 없습니다. page=%s", page)
        return None
    df_sel = df_page[selected].copy()
    if "routeName" in df_sel.columns:
        df_sel["routeName"] = df_sel["routeName"].astype(object).where(
            df_sel["routeName"].notna(), pd.NA
        )
        df_sel["routeName"] = df_sel["routeName"].astype(str).str.strip().replace({"": pd.NA, "None": pd.NA})
    return df_sel

first_df = process_page_json(js, 1)
if first_df is not None:
    frames.append(first_df)

In [ ]:
# 나머지 페이지 순회
for page in range(2, total_pages + 1):
    fj = fetch_json(page)
    if not fj:
        logging.error("페이지 %s 데이터를 가져오지 못해 스킵합니다.", page)
        continue
    df_sel = process_page_json(fj, page)
    if df_sel is not None:
        frames.append(df_sel)
    time.sleep(1 + random.random() * 0.5)

In [ ]:
# 결합
if frames:
    df = pd.concat(frames, ignore_index=True)
else:
    df = pd.DataFrame(columns=want_cols)

logging.info("수집 완료. 총 행: %s", len(df))

df

대성님꺼

In [ ]:
import os
import time
import math
import logging
import requests
import pandas as pd
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv

# 1. 환경 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
load_dotenv()

DATA_EX_KEY = os.getenv("DATA_EX_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")
DB_PORT = int(os.getenv("DB_PORT", "3306"))

# 3. 메인 실행 로직
df = fetch_api_data()

if df.empty:
    logging.error("데이터가 없습니다.")
else:
    try:
        conn = mysql.connector.connect(
            host=DB_HOST, user=DB_USER, password=DB_PASSWORD, 
            database=DB_NAME, port=DB_PORT, charset="utf8mb4"
        )
        cursor = conn.cursor()

        # DB 스키마: disel_price (int), gasoline_price (int), lpg_price (int)
        insert_sql = """
        INSERT INTO rest_area_gas 
        (restarea_name, route_name, lpgYn, gasoline_price, disel_price, lpg_price, svarAddr)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
        route_name=VALUES(route_name), lpgYn=VALUES(lpgYn),
        gasoline_price=VALUES(gasoline_price), disel_price=VALUES(disel_price),
        lpg_price=VALUES(lpg_price), svarAddr=VALUES(svarAddr)
        """

        rows_to_insert = []
        for _, row in df.iterrows():
            # [형변환 적용]
            # PK: serviceAreaName 사용
            name = row.get("serviceAreaName")
            if not name: continue

            # 가격 정보: safe_str_to_int 함수로 처리
            g_price = safe_str_to_int(row.get("gasolinePrice"))
            d_price = safe_str_to_int(row.get("diselPrice"))
            l_price = safe_str_to_int(row.get("lpgPrice"))

            # LPG 여부: string -> int (1 or 0)
            lpg_yn = 1 if str(row.get("lpgYn", "N")).upper() in ["Y", "1"] else 0

            rows_to_insert.append((
                str(name)[:200], 
                str(row.get("routeName", ""))[:100],
                lpg_yn, g_price, d_price, l_price,
                str(row.get("svarAddr", ""))[:50]
            ))

        # DB 일괄 저장
        for i in range(0, len(rows_to_insert), 500):
            chunk = rows_to_insert[i:i+500]
            cursor.executemany(insert_sql, chunk)
            conn.commit()
            logging.info(f"{i + len(chunk)}건 처리 완료")

        logging.info("정수 형변환 및 DB 업데이트 완료!")

    except Error as e:
        logging.error(f"DB 작업 오류: {e}")
    finally:
        if conn and conn.is_connected():
            cursor.close()
            conn.close()

df

sql 데이터 적재

In [ ]:
import os
import logging
import requests
import pandas as pd
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv

# 1. 환경 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
load_dotenv()

DATA_EX_KEY = os.getenv("DATA_EX_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")
DB_PORT = int(os.getenv("DB_PORT", "3306"))

# 2. 유틸 함수 정의
def safe_str_to_int(value):
    """문자열을 안전하게 정수로 변환"""
    try:
        return int(str(value).replace(",", "").strip())
    except (ValueError, TypeError):
        return None

def fetch_api_data():
    """API 호출 후 DataFrame 반환"""
    url = f"https://api.example.com/data?key={DATA_EX_KEY}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        return pd.DataFrame(data.get("list", []))
    except Exception as e:
        logging.error(f"API 호출 오류: {e}")
        return pd.DataFrame()

# 3. 메인 실행 로직
df = fetch_api_data()

if df.empty:
    logging.error("데이터가 없습니다.")
else:
    try:
        conn = mysql.connector.connect(
            host=DB_HOST, user=DB_USER, password=DB_PASSWORD,
            database=DB_NAME, port=DB_PORT, charset="utf8mb4"
        )
        cursor = conn.cursor()

        # DB 스키마: diesel_price (int), gasoline_price (int), lpg_price (int)
        insert_sql = """
        INSERT INTO rest_area_gas 
        (restarea_name, route_name, lpgYn, gasoline_price, diesel_price, lpg_price, svarAddr)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
        route_name=VALUES(route_name), lpgYn=VALUES(lpgYn),
        gasoline_price=VALUES(gasoline_price), diesel_price=VALUES(diesel_price),
        lpg_price=VALUES(lpg_price), svarAddr=VALUES(svarAddr)
        """

        rows_to_insert = []
        for _, row in df.iterrows():
            # PK: serviceAreaName 사용
            name = row.get("serviceAreaName")
            if not name:
                continue

            g_price = safe_str_to_int(row.get("gasolinePrice"))
            d_price = safe_str_to_int(row.get("diselPrice"))  # API에서 'diselPrice'로 내려올 수 있음
            l_price = safe_str_to_int(row.get("lpgPrice"))

            lpg_yn = 1 if str(row.get("lpgYn", "N")).upper() in ["Y", "1"] else 0

            rows_to_insert.append((
                str(name)[:200],
                str(row.get("routeName", ""))[:100],
                lpg_yn, g_price, d_price, l_price,
                str(row.get("svarAddr", ""))[:50]
            ))

        # DB 일괄 저장
        for i in range(0, len(rows_to_insert), 500):
            chunk = rows_to_insert[i:i+500]
            cursor.executemany(insert_sql, chunk)
            conn.commit()
            logging.info(f"{i + len(chunk)}건 처리 완료")

        logging.info("정수 형변환 및 DB 업데이트 완료!")

    except Error as e:
        logging.error(f"DB 작업 오류: {e}")
    finally:
        if conn and conn.is_connected():
            cursor.close()
            conn.close()

# 결과 확인용
print(df.head())

In [ ]:
import os
import time
import math
import logging
import requests
import random
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
load_dotenv()

DATA_EX_KEY = os.getenv("DATA_EX_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME_REST = os.getenv("DB_NAME")
DB_PORT = os.getenv("DB_PORT", "3306")

if not DATA_EX_KEY:
    raise RuntimeError("환경변수 DATA_EX_KEY가 설정되어 있지 않습니다.")
if not (DB_HOST and DB_USER and DB_PASSWORD and DB_NAME):
    raise RuntimeError("MySQL 접속을 위한 환경변수(DB_HOST, DB_USER, DB_PASSWORD, DB_NAME)가 모두 설정되어 있어야 합니다.")

# 크롤링 설정
base_url = "https://data.ex.co.kr/openapi/business/curStateStation"
per_page = 100
params = {"key": DATA_EX_KEY, "type": "json", "numOfRows": per_page, "pageNo": 1}

# 원하는 컬럼 목록, 프레임 리스트 초기화
want_cols = ["routeCode", "serviceAreaCode", "routeName", "lpgYn", "gasolinePrice", "diselPrice", "lpgPrice", "svarAddr"]
frames = []
print(want_cols)

# requests 세션 사용
rq = requests.Session()
rq.headers.update({"User-Agent": "MyCrawler/1.0 (+https://example.com)"})
def fetch_json(page, retries=2):
    params_local = params.copy()
    params_local["pageNo"] = page
    backoff = 1
    for attempt in range(retries + 1):
        try:
            rp = rq.get(base_url, params=params_local, timeout=10)
            rp.raise_for_status()
            return rp.json()
        except requests.RequestException as e:
            logging.warning("Request error page %s attempt %s: %s", page, attempt+1, e)
        except ValueError as e:
            logging.warning("JSON decode error page %s attempt %s: %s", page, attempt+1, e)
        time.sleep(backoff + random.random() * 0.5)
        backoff *= 2
    return None

# 총 개수 확인 (안전하게 처리)
js = fetch_json(1)
if not js:
    raise RuntimeError("첫 페이지 호출 실패로 진행 불가")
total_count = js.get("count") or js.get("totalCount") or 0
if not total_count:
    logging.info("count 정보가 없거나 0입니다. 첫 페이지 데이터만 처리합니다.")
    total_pages = 1
else:
    total_pages = math.ceil(int(total_count) / per_page)
logging.info("총 항목: %s, 페이지 수: %s, 페이지당: %s", total_count, total_pages, per_page)

# 첫 페이지 처리
def process_page_json(js, page):
    page_list = js.get("list") or []
    if not page_list:
        return None
    df_page = pd.json_normalize(page_list)
    df_page.columns = df_page.columns.str.strip()
    if "pageNo" not in df_page.columns or df_page["pageNo"].isna().all():
        df_page = df_page.copy()
        df_page["_page"] = page
    selected = [col for col in want_cols if col in df_page.columns]
    if "_page" in df_page.columns and "_page" not in selected:
        selected.append("_page")
    if not selected:
        logging.info("원하는 컬럼이 페이지에 없습니다. page=%s", page)
        return None
    df_sel = df_page[selected].copy()
    if "routeName" in df_sel.columns:
        df_sel["routeName"] = df_sel["routeName"].astype(object).where(
            df_sel["routeName"].notna(), pd.NA
        )
        df_sel["routeName"] = df_sel["routeName"].astype(str).str.strip().replace({"": pd.NA, "None": pd.NA})
    return df_sel

first_df = process_page_json(js, 1)
if first_df is not None:
    frames.append(first_df)

# 나머지 페이지 순회
for page in range(2, total_pages + 1):
    fj = fetch_json(page)
    if not fj:
        logging.error("페이지 %s 데이터를 가져오지 못해 스킵합니다.", page)
        continue
    df_sel = process_page_json(fj, page)
    if df_sel is not None:
        frames.append(df_sel)
    time.sleep(1 + random.random() * 0.5)

# 결합
if frames:
    df = pd.concat(frames, ignore_index=True)
else:
    df = pd.DataFrame(columns=want_cols)

logging.info("수집 완료. 총 행: %s", len(df))

df

In [ ]:
import os
import time
import math
import logging
import requests 
import random
import pandas as pd
from dotenv import load_dotenv
import mysql.connector
from mysql.connector import Error

# 1. 환경 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
load_dotenv()

DATA_EX_KEY = os.getenv("DATA_EX_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME_REST = os.getenv("DB_NAME")
DB_PORT = int(os.getenv("DB_PORT", "3306"))

if not DATA_EX_KEY:
    raise RuntimeError("환경변수 DATA_EX_KEY가 설정되어 있지 않습니다.")
if not (DB_HOST and DB_USER and DB_PASSWORD and DB_NAME):
    raise RuntimeError("MySQL 접속을 위한 환경변수(DB_HOST, DB_USER, DB_PASSWORD, DB_NAME)가 모두 설정되어 있어야 합니다.")

# 2. 크롤링 설정
base_url = "https://data.ex.co.kr/openapi/business/curStateStation"
per_page = 100
params = {"key": DATA_EX_KEY, "type": "json", "numOfRows": per_page, "pageNo": 1}

want_cols = ["serviceAreaName", "routeName", "lpgYn", "gasolinePrice", "diselPrice", "lpgPrice", "svarAddr"]
frames = []

rq = requests.Session()
rq.headers.update({"User-Agent": "MyCrawler/1.0 (+https://example.com)"})

def fetch_json(page, retries=2):
    params_local = params.copy()
    params_local["pageNo"] = page
    backoff = 1
    for attempt in range(retries + 1):
        try:
            rp = rq.get(base_url, params=params_local, timeout=10)
            rp.raise_for_status()
            return rp.json()
        except requests.RequestException as e:
            logging.warning("Request error page %s attempt %s: %s", page, attempt+1, e)
        except ValueError as e:
            logging.warning("JSON decode error page %s attempt %s: %s", page, attempt+1, e)
        time.sleep(backoff + random.random() * 0.5)
        backoff *= 2
    return None

# 3. 총 개수 확인
js = fetch_json(1)
if not js:
    raise RuntimeError("첫 페이지 호출 실패로 진행 불가")
total_count = js.get("count") or js.get("totalCount") or 0
total_pages = math.ceil(int(total_count) / per_page) if total_count else 1
logging.info("총 항목: %s, 페이지 수: %s, 페이지당: %s", total_count, total_pages, per_page)

def process_page_json(js, page):
    page_list = js.get("list") or []
    if not page_list:
        return None
    df_page = pd.json_normalize(page_list)
    df_page.columns = df_page.columns.str.strip()
    selected = [col for col in want_cols if col in df_page.columns]
    if not selected:
        logging.info("원하는 컬럼이 페이지에 없습니다. page=%s", page)
        return None
    df_sel = df_page[selected].copy()
    if "routeName" in df_sel.columns:
        df_sel["routeName"] = df_sel["routeName"].astype(str).str.strip().replace({"": pd.NA, "None": pd.NA})
    return df_sel

# 4. 데이터 수집
first_df = process_page_json(js, 1)
if first_df is not None:
    frames.append(first_df)

for page in range(2, total_pages + 1):
    fj = fetch_json(page)
    if not fj:
        logging.error("페이지 %s 데이터를 가져오지 못해 스킵합니다.", page)
        continue
    df_sel = process_page_json(fj, page)
    if df_sel is not None:
        frames.append(df_sel)
    time.sleep(1 + random.random() * 0.5)

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=want_cols)
logging.info("수집 완료. 총 행: %s", len(df))

# 5. 안전한 정수 변환
def safe_str_to_int(value):
    try:
        return int(str(value).replace(",", "").strip())
    except (ValueError, TypeError):
        return None

# 6. DB 연결 및 저장
try:
    conn = mysql.connector.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME,
        port=DB_PORT,
        charset="utf8mb4"
    )
    cursor = conn.cursor()

    insert_sql = """
    INSERT INTO rest_area_gas 
    (restarea_name, route_name, lpgYn, gasoline_price, disel_price, lpg_price, svarAddr)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
    route_name=VALUES(route_name), lpgYn=VALUES(lpgYn),
    gasoline_price=VALUES(gasoline_price), disel_price=VALUES(disel_price),
    lpg_price=VALUES(lpg_price), svarAddr=VALUES(svarAddr)
    """

    rows_to_insert = []
    for _, row in df.iterrows():
        name = row.get("serviceAreaName")
        if not name:
            continue
        rows_to_insert.append((
            str(name)[:200],
            str(row.get("routeName", ""))[:100],
            1 if str(row.get("lpgYn", "N")).upper() in ["Y", "1"] else 0,
            safe_str_to_int(row.get("gasolinePrice")),
            safe_str_to_int(row.get("diselPrice")),
            safe_str_to_int(row.get("lpgPrice")),
            str(row.get("svarAddr", ""))[:50]
        ))

    for i in range(0, len(rows_to_insert), 500):
        chunk = rows_to_insert[i:i+500]
        cursor.executemany(insert_sql, chunk)
        conn.commit()
        logging.info("%s건 처리 완료", i + len(chunk))

    logging.info("DB 업데이트 완료!")

except Error as e:
    logging.error(f"DB 작업 오류: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

In [ ]:
print(df.columns)   # 실제 컬럼명 확인
print(df.head())    # 데이터 확인

In [ ]:
import os
import time
import math
import logging
import requests
import random
import pandas as pd
from dotenv import load_dotenv
import mysql.connector
from mysql.connector import Error

# 1. 환경 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
load_dotenv()

DATA_EX_KEY = os.getenv("DATA_EX_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME_REST").strip()  # 주의: 코드 전반에서 DB_NAME으로 통일
DB_PORT = int(os.getenv("DB_PORT", "3306"))

if not DATA_EX_KEY:
    raise RuntimeError("환경변수 DATA_EX_KEY가 설정되어 있지 않습니다.")
if not (DB_HOST and DB_USER and DB_PASSWORD and DB_NAME):
    raise RuntimeError("MySQL 접속을 위한 환경변수(DB_HOST, DB_USER, DB_PASSWORD, DB_NAME)가 모두 설정되어 있어야 합니다.")

# 2. 크롤링 설정
base_url = "https://data.ex.co.kr/openapi/business/curStateStation"
per_page = 100
params = {"key": DATA_EX_KEY, "type": "json", "numOfRows": per_page, "pageNo": 1}

# 실제 API 응답에 따라 컬럼명이 다를 수 있으므로 여러 후보를 허용
# 최종 매핑은 df.columns를 확인한 뒤 결정
want_cols = ["serviceAreaName", "serviceAreaNm", "routeName", "lpgYn", "gasolinePrice", "diselPrice", "lpgPrice", "svarAddr"]
frames = []

rq = requests.Session()
rq.headers.update({"User-Agent": "MyCrawler/1.0 (+https://example.com)"})

def fetch_json(page, retries=2):
    params_local = params.copy()
    params_local["pageNo"] = page
    backoff = 1
    for attempt in range(retries + 1):
        try:
            rp = rq.get(base_url, params=params_local, timeout=10)
            rp.raise_for_status()
            return rp.json()
        except requests.RequestException as e:
            logging.warning("Request error page %s attempt %s: %s", page, attempt+1, e)
        except ValueError as e:
            logging.warning("JSON decode error page %s attempt %s: %s", page, attempt+1, e)
        time.sleep(backoff + random.random() * 0.5)
        backoff *= 2
    return None

# 3. 총 개수 확인
js = fetch_json(1)
if not js:
    raise RuntimeError("첫 페이지 호출 실패로 진행 불가")
total_count = js.get("count") or js.get("totalCount") or 0
total_pages = math.ceil(int(total_count) / per_page) if total_count else 1
logging.info("총 항목: %s, 페이지 수: %s, 페이지당: %s", total_count, total_pages, per_page)

def process_page_json(js, page):
    page_list = js.get("list") or []
    if not page_list:
        return None
    df_page = pd.json_normalize(page_list)
    df_page.columns = df_page.columns.str.strip()
    # 선택 가능한 컬럼 중 실제 존재하는 컬럼만 선택
    selected = [col for col in want_cols if col in df_page.columns]
    if not selected:
        logging.info("원하는 컬럼이 페이지에 없습니다. page=%s", page)
        return None
    df_sel = df_page[selected].copy()
    # routeName 정리
    if "routeName" in df_sel.columns:
        df_sel["routeName"] = df_sel["routeName"].astype(str).str.strip().replace({"": pd.NA, "None": pd.NA})
    return df_sel

# 4. 데이터 수집
first_df = process_page_json(js, 1)
if first_df is not None:
    frames.append(first_df)

for page in range(2, total_pages + 1):
    fj = fetch_json(page)
    if not fj:
        logging.error("페이지 %s 데이터를 가져오지 못해 스킵합니다.", page)
        continue
    df_sel = process_page_json(fj, page)
    if df_sel is not None:
        frames.append(df_sel)
    time.sleep(1 + random.random() * 0.5)

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=want_cols)
logging.info("수집 완료. 총 행: %s", len(df))

# 4.5 데이터 확인 (중요)
# 실제 컬럼명과 샘플을 로그로 확인해서 매핑 문제를 방지
logging.info("DataFrame columns: %s", list(df.columns))
logging.info("DataFrame sample:\n%s", df.head(5).to_string(index=False))

# 5. 안전한 정수 변환
def safe_str_to_int(value):
    try:
        if value is None:
            return None
        s = str(value).strip()
        if s == "" or s.lower() == "none":
            return None
        return int(s.replace(",", ""))
    except (ValueError, TypeError):
        return None

# 6. DB 연결 및 (필요시) DB/테이블 생성
conn = None
try:
    # 먼저 서버에 접속 (database 미지정)하여 DB 존재 여부 확인 및 생성
    admin_conn = mysql.connector.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT,
        charset="utf8mb4"
    )
    admin_cursor = admin_conn.cursor()
    admin_cursor.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}` CHARACTER SET utf8mb4 COLLATE utf8mb4_general_ci;")
    admin_conn.commit()
    admin_cursor.close()
    admin_conn.close()
    logging.info("데이터베이스 존재 확인 또는 생성 완료: %s", DB_NAME)

    # 이제 실제 DB에 연결
    conn = mysql.connector.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME,
        port=DB_PORT,
        charset="utf8mb4"
    )
    cursor = conn.cursor()

    # 테이블이 없으면 생성 (INT(11) 같은 display width 사용하지 않음)
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS rest_area_gas (
        restarea_name VARCHAR(200) NOT NULL,
        route_name VARCHAR(100),
        lpgYn TINYINT,
        gasoline_price INT,
        disel_price INT,
        lpg_price INT,
        svarAddr VARCHAR(50),
        PRIMARY KEY (restarea_name)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """
    cursor.execute(create_table_sql)
    conn.commit()
    logging.info("테이블 존재 확인 또는 생성 완료: rest_area_gas")

    # 7. 컬럼 매핑: API 컬럼명이 여러 후보일 수 있으므로 우선순위로 추출
    def get_service_area_name(row):
        # 후보 키들 중 첫 번째로 존재하는 값을 반환
        for key in ("serviceAreaName", "serviceAreaNm", "service_area_name"):
            if key in row and pd.notna(row.get(key)):
                return row.get(key)
        # DataFrame에 컬럼이 없을 수도 있으므로 try direct access
        for key in row.index:
            if key.lower() in ("serviceareaname", "serviceareanm", "service_area_name"):
                return row.get(key)
        return None

    # 8. 삽입할 행 준비
    rows_to_insert = []
    for _, row in df.iterrows():
        name = get_service_area_name(row)
        if not name:
            # PK가 없으면 삽입 불가, 로그 남기기
            logging.warning("서비스 구역 이름 누락으로 스킵: row index=%s", _)
            continue

        route_name = None
        # routeName 후보 확인
        for key in ("routeName", "route_name"):
            if key in row and pd.notna(row.get(key)):
                route_name = str(row.get(key)).strip()
                break

        lpgYn_val = None
        if "lpgYn" in row and pd.notna(row.get("lpgYn")):
            lpgYn_val = 1 if str(row.get("lpgYn")).strip().upper() in ("Y", "1", "TRUE", "T") else 0

        gasoline_price = None
        for key in ("gasolinePrice", "gasoline_price"):
            if key in row and pd.notna(row.get(key)):
                gasoline_price = safe_str_to_int(row.get(key))
                break

        disel_price = None
        for key in ("diselPrice", "dieselPrice", "disel_price", "diesel_price"):
            if key in row and pd.notna(row.get(key)):
                disel_price = safe_str_to_int(row.get(key))
                break

        lpg_price = None
        for key in ("lpgPrice", "lpg_price"):
            if key in row and pd.notna(row.get(key)):
                lpg_price = safe_str_to_int(row.get(key))
                break

        svarAddr = None
        for key in ("svarAddr", "svar_addr", "addr"):
            if key in row and pd.notna(row.get(key)):
                svarAddr = str(row.get(key)).strip()
                break

        rows_to_insert.append((
            str(name)[:200],
            str(route_name)[:100] if route_name is not None else None,
            lpgYn_val,
            gasoline_price,
            disel_price,
            lpg_price,
            str(svarAddr)[:50] if svarAddr is not None else None
        ))

    logging.info("DB에 삽입할 준비된 행 수: %s", len(rows_to_insert))

    # 9. 삽입 (배치)
    insert_sql = """
    INSERT INTO rest_area_gas
    (restarea_name, route_name, lpgYn, gasoline_price, disel_price, lpg_price, svarAddr)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
    route_name=VALUES(route_name), lpgYn=VALUES(lpgYn),
    gasoline_price=VALUES(gasoline_price), disel_price=VALUES(disel_price),
    lpg_price=VALUES(lpg_price), svarAddr=VALUES(svarAddr)
    """

    inserted = 0
    for i in range(0, len(rows_to_insert), 500):
        chunk = rows_to_insert[i:i+500]
        cursor.executemany(insert_sql, chunk)
        conn.commit()
        inserted += len(chunk)
        logging.info("%s건 처리 완료", i + len(chunk))

    logging.info("총 삽입 또는 업데이트된 행 수: %s", inserted)

except Error as e:
    logging.error("DB 작업 오류: %s", e)
except Exception as e:
    logging.error("예상치 못한 오류: %s", e)
finally:
    if 'conn' in locals() and conn is not None and conn.is_connected():
        try:
            cursor.close()
        except Exception:
            pass
        conn.close()
        logging.info("DB 연결 종료")

In [ ]:
import os, time, math, logging, requests, random, pandas as pd
from dotenv import load_dotenv
import mysql.connector
from mysql.connector import Error

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
load_dotenv()

DATA_EX_KEY = os.getenv("DATA_EX_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME_REST").strip()
DB_PORT = int(os.getenv("DB_PORT", "3306"))

logging.info("Using DB config: %s@%s:%s DB=%s", DB_USER, DB_HOST, DB_PORT, DB_NAME)

if not DATA_EX_KEY:
    raise RuntimeError("DATA_EX_KEY not set")
if not (DB_HOST and DB_USER and DB_PASSWORD and DB_NAME):
    raise RuntimeError("DB env vars missing")

# --- (크롤링 부분은 기존과 동일) ---
base_url = "https://data.ex.co.kr/openapi/business/curStateStation"
per_page = 100
params = {"key": DATA_EX_KEY, "type": "json", "numOfRows": per_page, "pageNo": 1}
want_cols = ["serviceAreaName", "serviceAreaNm", "routeName", "lpgYn", "gasolinePrice", "diselPrice", "lpgPrice", "svarAddr"]
frames = []
rq = requests.Session()
rq.headers.update({"User-Agent": "MyCrawler/1.0"})

def fetch_json(page, retries=2):
    params_local = params.copy(); params_local["pageNo"] = page
    backoff = 1
    for attempt in range(retries+1):
        try:
            rp = rq.get(base_url, params=params_local, timeout=10)
            rp.raise_for_status()
            return rp.json()
        except Exception as e:
            logging.warning("fetch_json page %s attempt %s error: %s", page, attempt+1, e)
        time.sleep(backoff + random.random()*0.5); backoff *= 2
    return None

js = fetch_json(1)
if not js:
    raise RuntimeError("첫 페이지 호출 실패")
total_count = js.get("count") or js.get("totalCount") or 0
total_pages = math.ceil(int(total_count)/per_page) if total_count else 1
logging.info("총 항목: %s, 페이지 수: %s", total_count, total_pages)

def process_page_json(js):
    page_list = js.get("list") or []
    if not page_list: return None
    df_page = pd.json_normalize(page_list)
    df_page.columns = df_page.columns.str.strip()
    selected = [c for c in want_cols if c in df_page.columns]
    if not selected: return None
    df_sel = df_page[selected].copy()
    if "routeName" in df_sel.columns:
        df_sel["routeName"] = df_sel["routeName"].astype(str).str.strip().replace({"": pd.NA, "None": pd.NA})
    return df_sel

first_df = process_page_json(js)
if first_df is not None: frames.append(first_df)
for p in range(2, total_pages+1):
    fj = fetch_json(p)
    if not fj:
        logging.error("페이지 %s 실패", p); continue
    df_sel = process_page_json(fj)
    if df_sel is not None: frames.append(df_sel)
    time.sleep(1 + random.random()*0.5)

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=want_cols)
logging.info("수집 완료. 총 행: %s", len(df))
logging.info("DataFrame columns: %s", list(df.columns))
logging.info("DataFrame sample:\n%s", df.head(10).to_string(index=False))

def safe_str_to_int(v):
    try:
        if v is None: return None
        s = str(v).strip()
        if s == "" or s.lower() == "none": return None
        return int(s.replace(",", ""))
    except:
        return None

# --- DB 연결 및 삽입 전 권한/연결 테스트 ---
conn = None
try:
    # 1) 서버 접속(관리용)으로 DB 존재 확인
    admin = mysql.connector.connect(host=DB_HOST, user=DB_USER, password=DB_PASSWORD, port=DB_PORT)
    admin_cursor = admin.cursor()
    admin_cursor.execute("SELECT VERSION();")
    logging.info("MySQL server version: %s", admin_cursor.fetchone())
    # 권한 확인 (로그에 남김)
    try:
        admin_cursor.execute(f"SHOW GRANTS FOR '{DB_USER}'@'%'")
        grants = admin_cursor.fetchall()
        logging.info("Grants for user: %s", grants)
    except Exception as e:
        logging.warning("SHOW GRANTS failed: %s", e)
    admin_cursor.close(); admin.close()

    # 2) 실제 DB 연결
    conn = mysql.connector.connect(host=DB_HOST, user=DB_USER, password=DB_PASSWORD, database=DB_NAME, port=DB_PORT, charset="utf8mb4")
    cursor = conn.cursor()

    # 3) 테이블 존재 확인 (생성)
    create_sql = """
    CREATE TABLE IF NOT EXISTS rest_area_gas (
        restarea_name VARCHAR(200) NOT NULL,
        route_name VARCHAR(100),
        lpgYn TINYINT,
        gasoline_price INT,
        disel_price INT,
        lpg_price INT,
        svarAddr VARCHAR(50),
        PRIMARY KEY (restarea_name)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """
    cursor.execute(create_sql); conn.commit()
    logging.info("테이블 확인/생성 완료")

    # 4) 삽입 준비 (매핑 확인)
    def get_name(row):
        for k in ("serviceAreaName","serviceAreaNm"):
            if k in row and pd.notna(row.get(k)): return row.get(k)
        for k in row.index:
            if k.lower() in ("serviceareaname","serviceareanm"): return row.get(k)
        return None

    rows = []
    skipped = 0
    for idx, row in df.iterrows():
        name = get_name(row)
        if not name:
            skipped += 1
            logging.debug("skip idx=%s no name row=%s", idx, row.to_dict())
            continue
        route = row.get("routeName") if "routeName" in row else None
        lpgYn = 1 if ("lpgYn" in row and pd.notna(row.get("lpgYn")) and str(row.get("lpgYn")).strip().upper() in ("Y","1","TRUE")) else 0
        gp = safe_str_to_int(row.get("gasolinePrice")) if "gasolinePrice" in row else None
        dp = safe_str_to_int(row.get("diselPrice")) if "diselPrice" in row else None
        lp = safe_str_to_int(row.get("lpgPrice")) if "lpgPrice" in row else None
        addr = row.get("svarAddr") if "svarAddr" in row else None
        rows.append((str(name)[:200], str(route)[:100] if route is not None else None, lpgYn, gp, dp, lp, str(addr)[:50] if addr is not None else None))

    logging.info("Prepared rows=%s skipped=%s", len(rows), skipped)
    logging.info("Sample prepared rows: %s", rows[:5])

    # 5) 수동 테스트: 한 건만 먼저 삽입해보기 (권한/문제 확인용)
    if rows:
        test_row = rows[0]
        try:
            cursor.execute("INSERT INTO rest_area_gas (restarea_name, route_name) VALUES (%s, %s) ON DUPLICATE KEY UPDATE route_name=VALUES(route_name)", (test_row[0], test_row[1]))
            conn.commit()
            logging.info("Test insert OK for %s", test_row[0])
            # 삭제 테스트 데이터(원치 않으면 주석)
            cursor.execute("DELETE FROM rest_area_gas WHERE restarea_name=%s", (test_row[0],))
            conn.commit()
        except Exception as e:
            logging.error("Test insert failed: %s", e)
            raise

    # 6) 배치 삽입
    insert_sql = """
    INSERT INTO rest_area_gas (restarea_name, route_name, lpgYn, gasoline_price, disel_price, lpg_price, svarAddr)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE route_name=VALUES(route_name), lpgYn=VALUES(lpgYn),
    gasoline_price=VALUES(gasoline_price), disel_price=VALUES(disel_price),
    lpg_price=VALUES(lpg_price), svarAddr=VALUES(svarAddr)
    """
    total_inserted = 0
    for i in range(0, len(rows), 500):
        chunk = rows[i:i+500]
        cursor.executemany(insert_sql, chunk)
        conn.commit()
        total_inserted += len(chunk)
        logging.info("Chunk inserted: %s rows (total %s)", len(chunk), total_inserted)

    logging.info("Finished. total inserted/updated: %s", total_inserted)

except Error as e:
    logging.exception("MySQL Error: %s", e)
except Exception as e:
    logging.exception("Unexpected Error: %s", e)
finally:
    if 'conn' in locals() and conn is not None and conn.is_connected():
        try: cursor.close()
        except: pass
        conn.close()
        logging.info("DB connection closed")

In [ ]:
import os
import time
import math
import logging
import requests
import random
import pandas as pd
from dotenv import load_dotenv
import mysql.connector
from mysql.connector import Error

# 1. 환경 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
load_dotenv()

DATA_EX_KEY = os.getenv("DATA_EX_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME").strip()
DB_PORT = int(os.getenv("DB_PORT", "3306"))

if not DATA_EX_KEY:
    raise RuntimeError("환경변수 DATA_EX_KEY가 설정되어 있지 않습니다.")
if not (DB_HOST and DB_USER and DB_PASSWORD and DB_NAME):
    raise RuntimeError("MySQL 접속을 위한 환경변수(DB_HOST, DB_USER, DB_PASSWORD, DB_NAME)가 모두 설정되어 있어야 합니다.")

# 2. 크롤링 설정
# 사용하신 엔드포인트로 변경
base_url = "https://data.ex.co.kr/openapi/business/serviceAreaRoute"
per_page = 100
params = {"key": DATA_EX_KEY, "type": "json", "numOfRows": per_page, "pageNo": 1}

# 후보 컬럼 목록
want_cols = ["serviceAreaName", "serviceAreaNm", "routeName", "lpgYn", "gasolinePrice", "diselPrice", "lpgPrice", "svarAddr"]
frames = []

rq = requests.Session()
rq.headers.update({"User-Agent": "MyCrawler/1.0 (+https://example.com)"})

def fetch_json(page, retries=2):
    params_local = params.copy()
    params_local["pageNo"] = page
    backoff = 1
    for attempt in range(retries + 1):
        try:
            rp = rq.get(base_url, params=params_local, timeout=10)
            rp.raise_for_status()
            return rp.json()
        except requests.RequestException as e:
            logging.warning("Request error page %s attempt %s: %s", page, attempt+1, e)
        except ValueError as e:
            logging.warning("JSON decode error page %s attempt %s: %s", page, attempt+1, e)
        time.sleep(backoff + random.random() * 0.5)
        backoff *= 2
    return None

# 3. 총 개수 확인 (방어적 처리)
js = fetch_json(1)
if not js:
    raise RuntimeError("첫 페이지 호출 실패로 진행 불가")

# API마다 총개수 키가 다를 수 있으므로 여러 후보 확인
total_count = 0
for key in ("count", "totalCount", "total_count", "total"):
    if key in js and js.get(key) is not None:
        try:
            total_count = int(js.get(key))
            break
        except Exception:
            continue

total_pages = math.ceil(total_count / per_page) if total_count else 1
logging.info("총 항목: %s, 페이지 수: %s, 페이지당: %s", total_count, total_pages, per_page)

def process_page_json(js):
    # list 키 후보 확인
    page_list = None
    for key in ("list", "items", "data", "rows"):
        if key in js and js.get(key) is not None:
            page_list = js.get(key)
            break
    if not page_list:
        return None

    df_page = pd.json_normalize(page_list)
    # 핵심 수정: columns를 함수처럼 호출하면 오류 발생 -> strip으로 정리
    df_page.columns = df_page.columns.str.strip()
    # 선택 가능한 컬럼 중 실제 존재하는 컬럼만 선택
    selected = [col for col in want_cols if col in df_page.columns]
    if not selected:
        logging.info("원하는 컬럼이 페이지에 없습니다. available columns=%s", list(df_page.columns))
        return None
    df_sel = df_page[selected].copy()
    if "routeName" in df_sel.columns:
        df_sel["routeName"] = df_sel["routeName"].astype(str).str.strip().replace({"": pd.NA, "None": pd.NA})
    return df_sel

# 4. 데이터 수집
first_df = process_page_json(js)
if first_df is not None:
    frames.append(first_df)

for page in range(2, total_pages + 1):
    fj = fetch_json(page)
    if not fj:
        logging.error("페이지 %s 데이터를 가져오지 못해 스킵합니다.", page)
        continue
    df_sel = process_page_json(fj)
    if df_sel is not None:
        frames.append(df_sel)
    time.sleep(1 + random.random() * 0.5)

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=want_cols)
logging.info("수집 완료. 총 행: %s", len(df))

# 4.5 데이터 확인
logging.info("DataFrame columns: %s", list(df.columns))
logging.info("DataFrame sample:\n%s", df.head(10).to_string(index=False))

# 5. 안전한 정수 변환
def safe_str_to_int(value):
    try:
        if value is None:
            return None
        s = str(value).strip()
        if s == "" or s.lower() == "none":
            return None
        return int(s.replace(",", ""))
    except (ValueError, TypeError):
        return None

# 6. DB 연결 및 삽입 (기존 로직 사용)
conn = None
try:
    admin_conn = mysql.connector.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT,
        charset="utf8mb4"
    )
    admin_cursor = admin_conn.cursor()
    admin_cursor.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}` CHARACTER SET utf8mb4 COLLATE utf8mb4_general_ci;")
    admin_conn.commit()
    admin_cursor.close()
    admin_conn.close()
    logging.info("데이터베이스 존재 확인 또는 생성 완료: %s", DB_NAME)

    conn = mysql.connector.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME,
        port=DB_PORT,
        charset="utf8mb4"
    )
    cursor = conn.cursor()

    create_table_sql = """
    CREATE TABLE IF NOT EXISTS rest_area_gas (
        restarea_name VARCHAR(200) NOT NULL,
        route_name VARCHAR(100),
        lpgYn TINYINT,
        gasoline_price INT,
        disel_price INT,
        lpg_price INT,
        svarAddr VARCHAR(50),
        PRIMARY KEY (restarea_name)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """
    cursor.execute(create_table_sql)
    conn.commit()
    logging.info("테이블 존재 확인 또는 생성 완료: rest_area_gas")

    # 매핑 및 삽입 준비
    def get_service_area_name(row):
        for key in ("serviceAreaName", "serviceAreaNm", "service_area_name"):
            if key in row and pd.notna(row.get(key)):
                return row.get(key)
        for key in row.index:
            if key.lower() in ("serviceareaname", "serviceareanm", "service_area_name"):
                return row.get(key)
        return None

    rows_to_insert = []
    for idx, row in df.iterrows():
        name = get_service_area_name(row)
        if not name:
            logging.warning("서비스 구역 이름 누락으로 스킵: row index=%s", idx)
            continue

        route_name = None
        for key in ("routeName", "route_name"):
            if key in row and pd.notna(row.get(key)):
                route_name = str(row.get(key)).strip()
                break

        lpgYn_val = None
        if "lpgYn" in row and pd.notna(row.get("lpgYn")):
            lpgYn_val = 1 if str(row.get("lpgYn")).strip().upper() in ("Y", "1", "TRUE", "T") else 0

        gasoline_price = None
        for key in ("gasolinePrice", "gasoline_price"):
            if key in row and pd.notna(row.get(key)):
                gasoline_price = safe_str_to_int(row.get(key))
                break

        disel_price = None
        for key in ("diselPrice", "dieselPrice", "disel_price", "diesel_price"):
            if key in row and pd.notna(row.get(key)):
                disel_price = safe_str_to_int(row.get(key))
                break

        lpg_price = None
        for key in ("lpgPrice", "lpg_price"):
            if key in row and pd.notna(row.get(key)):
                lpg_price = safe_str_to_int(row.get(key))
                break

        svarAddr = None
        for key in ("svarAddr", "svar_addr", "addr"):
            if key in row and pd.notna(row.get(key)):
                svarAddr = str(row.get(key)).strip()
                break

        rows_to_insert.append((
            str(name)[:200],
            str(route_name)[:100] if route_name is not None else None,
            lpgYn_val,
            gasoline_price,
            disel_price,
            lpg_price,
            str(svarAddr)[:50] if svarAddr is not None else None
        ))

    logging.info("DB에 삽입할 준비된 행 수: %s", len(rows_to_insert))

    insert_sql = """
    INSERT INTO rest_area_gas
    (restarea_name, route_name, lpgYn, gasoline_price, disel_price, lpg_price, svarAddr)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
    route_name=VALUES(route_name), lpgYn=VALUES(lpgYn),
    gasoline_price=VALUES(gasoline_price), disel_price=VALUES(disel_price),
    lpg_price=VALUES(lpg_price), svarAddr=VALUES(svarAddr)
    """

    inserted = 0
    for i in range(0, len(rows_to_insert), 500):
        chunk = rows_to_insert[i:i+500]
        cursor.executemany(insert_sql, chunk)
        conn.commit()
        inserted += len(chunk)
        logging.info("%s건 처리 완료", i + len(chunk))

    logging.info("총 삽입 또는 업데이트된 행 수: %s", inserted)

except Error as e:
    logging.exception("DB 작업 오류: %s", e)
except Exception as e:
    logging.exception("예상치 못한 오류: %s", e)
finally:
    if 'conn' in locals() and conn is not None and conn.is_connected():
        try:
            cursor.close()
        except Exception:
            pass
        conn.close()
        logging.info("DB 연결 종료")

In [ ]:
import os
import time
import logging
import requests
import pandas as pd
from dotenv import load_dotenv
import mysql.connector
from mysql.connector import Error

# -----------------------
# 환경 설정
# -----------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
load_dotenv()

DATA_EX_KEY = os.getenv("DATA_EX_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = (os.getenv("DB_NAME") or os.getenv("DB_NAME_REST") or "").strip()
DB_PORT = int(os.getenv("DB_PORT", "3306"))

if not DATA_EX_KEY:
    raise RuntimeError("환경변수 DATA_EX_KEY가 설정되어 있지 않습니다.")
if not (DB_HOST and DB_USER and DB_PASSWORD and DB_NAME):
    raise RuntimeError("MySQL 접속을 위한 환경변수(DB_HOST, DB_USER, DB_PASSWORD, DB_NAME)가 모두 설정되어 있어야 합니다.")

# -----------------------
# API 및 컬럼 설정
# -----------------------
base_url = "https://data.ex.co.kr/openapi/business/serviceAreaRoute"
per_page = 100
params_base = {"key": DATA_EX_KEY, "type": "json", "numOfRows": per_page}

# 사용자 제공 필드 목록 (요청하신 필드명 반영)
want_cols = [
    "routeCode", "serviceAreaCode", "routeName", "direction", "serviceAreaName",
    "telNo", "convenience", "brand", "maintenanceYn", "truckSaYn",
    "numOfRows", "pageNo", "serviceAreaCode2", "svarAddr"
]

# -----------------------
# 페이지 순회 (total count 없이, 빈 페이지 나오면 종료)
# -----------------------
def extract_page_list(json_obj):
    for key in ("list", "items", "data", "rows"):
        if key in json_obj and json_obj.get(key) is not None:
            return json_obj.get(key)
    return None

def process_page_list(page_list):
    if not page_list:
        return None
    df_page = pd.json_normalize(page_list)
    df_page.columns = df_page.columns.str.strip()
    selected = [c for c in want_cols if c in df_page.columns]
    if not selected:
        logging.info("원하는 컬럼이 페이지에 없습니다. available columns=%s", list(df_page.columns))
        return None
    df_sel = df_page[selected].copy()
    # 문자열 정리: 공백/None/<NA> 처리
    for col in df_sel.columns:
        if df_sel[col].dtype == object:
            df_sel[col] = df_sel[col].where(~df_sel[col].isna(), None)
            df_sel[col] = df_sel[col].astype(object).apply(lambda v: None if v is None else str(v).strip())
            df_sel[col] = df_sel[col].replace({"": None, "None": None, "<NA>": None})
    return df_sel

frames = []
page = 1
while True:
    params = params_base.copy()
    params["pageNo"] = page
    try:
        resp = requests.get(base_url, params=params, timeout=15, headers={"User-Agent": "DataCollector/1.0"})
        resp.raise_for_status()
        page_json = resp.json()
    except requests.RequestException as e:
        logging.warning("페이지 %s 요청 실패: %s (재시도 1회)", page, e)
        time.sleep(1.0)
        try:
            resp = requests.get(base_url, params=params, timeout=15, headers={"User-Agent": "DataCollector/1.0"})
            resp.raise_for_status()
            page_json = resp.json()
        except Exception as e2:
            logging.error("페이지 %s 재시도 실패: %s. 수집 종료.", page, e2)
            break

    page_list = extract_page_list(page_json)
    if not page_list:
        logging.info("페이지 %s에 데이터 없음 — 수집 종료", page)
        break

    df_sel = process_page_list(page_list)
    if df_sel is not None:
        frames.append(df_sel)
        logging.info("페이지 %s 수집 완료. 행 수: %s", page, len(df_sel))
    else:
        logging.info("페이지 %s에서 원하는 컬럼이 없어 스킵", page)

    page += 1
    time.sleep(1.0)

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=want_cols)
logging.info("수집 완료. 총 행: %s", len(df))

# 디버그: 컬럼 및 샘플 출력
logging.info("=== DataFrame columns ===")
logging.info("%s", list(df.columns))
logging.info("=== DataFrame sample (first 10 rows) ===")
logging.info("\n%s", df.head(10).to_string(index=False))

# -----------------------
# 유틸리티: 안전한 변환
# -----------------------
def safe_str_from_row(row, key, maxlen=None):
    if key not in row:
        return None
    val = row.get(key)
    if pd.isna(val) or val is None:
        return None
    s = str(val).strip()
    if s == "" or s.lower() == "none" or s == "<NA>":
        return None
    return s[:maxlen] if maxlen else s

def safe_int_from_row(row, key):
    val = row.get(key) if key in row else None
    if pd.isna(val) or val is None:
        return None
    s = str(val).strip().replace(",", "")
    if s == "" or s.lower() == "none":
        return None
    try:
        return int(s)
    except Exception:
        return None

# -----------------------
# DB 연결, 테이블 생성 및 삽입
# -----------------------
conn = None
cursor = None
try:
    # 관리자 연결: DB 존재 확인/생성
    admin_conn = mysql.connector.connect(
        host=DB_HOST, user=DB_USER, password=DB_PASSWORD, port=DB_PORT, charset="utf8mb4"
    )
    admin_cursor = admin_conn.cursor()
    admin_cursor.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}` CHARACTER SET utf8mb4 COLLATE utf8mb4_general_ci;")
    admin_conn.commit()
    admin_cursor.close()
    admin_conn.close()
    logging.info("데이터베이스 확인/생성 완료: %s", DB_NAME)

    # 실제 DB 연결
    conn = mysql.connector.connect(
        host=DB_HOST, user=DB_USER, password=DB_PASSWORD, database=DB_NAME, port=DB_PORT, charset="utf8mb4"
    )
    cursor = conn.cursor()

    # 테이블 생성: 코드에서 사용하는 컬럼명(스네이크 케이스)으로 생성
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS rest_area_gas (
        restarea_name VARCHAR(200) NOT NULL,
        route_name VARCHAR(100),
        route_code VARCHAR(100),
        service_area_code VARCHAR(100),
        direction VARCHAR(100),
        tel_no VARCHAR(100),
        convenience TEXT,
        brand VARCHAR(100),
        maintenance_yn VARCHAR(10),
        truck_sa_yn VARCHAR(10),
        service_area_code2 VARCHAR(100),
        svar_addr VARCHAR(255),
        lpgYn TINYINT,
        gasoline_price INT,
        disel_price INT,
        lpg_price INT,
        PRIMARY KEY (restarea_name)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """
    cursor.execute(create_table_sql)
    conn.commit()
    logging.info("테이블 확인/생성 완료: rest_area_gas")

    # 삽입 준비: DataFrame 컬럼명을 기준으로 안전하게 추출
    rows_to_insert = []
    skipped = 0
    for idx, row in df.iterrows():
        # PK 결정: serviceAreaName 우선
        name = safe_str_from_row(row, "serviceAreaName", 200) or safe_str_from_row(row, "serviceAreaNm", 200)
        if not name:
            skipped += 1
            logging.debug("skip idx=%s no serviceAreaName", idx)
            continue

        route_name = safe_str_from_row(row, "routeName", 100)
        route_code = safe_str_from_row(row, "routeCode", 100)
        service_area_code = safe_str_from_row(row, "serviceAreaCode", 100)
        direction = safe_str_from_row(row, "direction", 100)
        tel_no = safe_str_from_row(row, "telNo", 100)
        convenience = safe_str_from_row(row, "convenience")  # TEXT
        brand = safe_str_from_row(row, "brand", 100)
        maintenance_yn = safe_str_from_row(row, "maintenanceYn", 10)
        truck_sa_yn = safe_str_from_row(row, "truckSaYn", 10)
        service_area_code2 = safe_str_from_row(row, "serviceAreaCode2", 100)
        svar_addr = safe_str_from_row(row, "svarAddr", 255)

        # 가격/연료 필드는 이 엔드포인트에 없으므로 None 처리
        lpgYn_val = None
        gasoline_price = None
        disel_price = None
        lpg_price = None

        rows_to_insert.append((
            name,
            route_name,
            route_code,
            service_area_code,
            direction,
            tel_no,
            convenience,
            brand,
            maintenance_yn,
            truck_sa_yn,
            service_area_code2,
            svar_addr,
            lpgYn_val,
            gasoline_price,
            disel_price,
            lpg_price
        ))

    logging.info("Prepared rows=%s skipped=%s", len(rows_to_insert), skipped)
    logging.info("Sample prepared rows: %s", rows_to_insert[:5])

    # 권한/삽입 테스트: 한 건만 먼저 삽입해보기
    if rows_to_insert:
        test_row = rows_to_insert[0]
        try:
            cursor.execute(
                "INSERT INTO rest_area_gas (restarea_name, route_name, route_code) VALUES (%s, %s, %s) "
                "ON DUPLICATE KEY UPDATE route_name=VALUES(route_name), route_code=VALUES(route_code)",
                (test_row[0], test_row[1], test_row[2])
            )
            conn.commit()
            logging.info("Test insert OK for %s", test_row[0])
            # 테스트 데이터 삭제 (원치 않으면 주석 처리)
            cursor.execute("DELETE FROM rest_area_gas WHERE restarea_name=%s", (test_row[0],))
            conn.commit()
        except Exception as e:
            logging.exception("Test insert failed: %s", e)
            raise

    # 배치 삽입 (컬럼 순서와 튜플 순서 일치)
    insert_sql = """
    INSERT INTO rest_area_gas (
        restarea_name, route_name, route_code, service_area_code, direction,
        tel_no, convenience, brand, maintenance_yn, truck_sa_yn, service_area_code2, svar_addr,
        lpgYn, gasoline_price, disel_price, lpg_price
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        route_name=VALUES(route_name), route_code=VALUES(route_code), service_area_code=VALUES(service_area_code),
        direction=VALUES(direction), tel_no=VALUES(tel_no), convenience=VALUES(convenience),
        brand=VALUES(brand), maintenance_yn=VALUES(maintenance_yn), truck_sa_yn=VALUES(truck_sa_yn),
        service_area_code2=VALUES(service_area_code2), svar_addr=VALUES(svar_addr),
        lpgYn=VALUES(lpgYn), gasoline_price=VALUES(gasoline_price),
        disel_price=VALUES(disel_price), lpg_price=VALUES(lpg_price)
    """
    total_inserted = 0
    for i in range(0, len(rows_to_insert), 500):
        chunk = rows_to_insert[i:i+500]
        cursor.executemany(insert_sql, chunk)
        conn.commit()
        total_inserted += len(chunk)
        logging.info("Chunk inserted: %s rows (total %s)", len(chunk), total_inserted)

    logging.info("완료: 총 삽입/업데이트된 행 수: %s", total_inserted)

except Error as e:
    logging.exception("MySQL Error: %s", e)
except Exception as e:
    logging.exception("Unexpected Error: %s", e)
finally:
    if cursor is not None:
        try:
            cursor.close()
        except Exception:
            pass
    if conn is not None and conn.is_connected():
        conn.close()
        logging.info("DB 연결 종료")